### Importamos las librerias necesarias y importamos la demanda

In [109]:
!pip install pyomo
!pip install pulp
!conda install -c conda-forge glpk -y
!pip install highspy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 18.4 MB/s eta 0:00:00m eta 0:00:010:01:01
Channels:
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: done

# All requested packages already installed.



In [60]:
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
import math

In [72]:
# -------------------------------------------------------------
# FASE 1: LIMPIEZA DE DATOS (Todos los SKUs)
# -------------------------------------------------------------
df = pd.read_csv('historial_ventas_lavandina.csv')

df['Fecha'] = pd.to_datetime(df['Fecha'])
df['Mes_Anio'] = df['Fecha'].dt.to_period('M')

# Agrupamos sumando los litros por mes y por SKU
df_agrupado = df.groupby(['Mes_Anio', 'SKU'])['Litros_Vendidos'].sum().reset_index()



### Analizamos el pronostico, de cada sku

In [90]:
# -------------------------------------------------------------
# FASE 2: MOTOR DE PRONÓSTICO BATCH
# -------------------------------------------------------------
# Vamos a guardar la demanda del mes que queremos planificar (Noviembre 2026), formando un listado con sky y su respectiva demanda
# -------------------------------------------------------------

mes_objetivo = '2026-11-01'
demanda_noviembre = {}

print(f"--- PRONÓSTICO DE DEMANDA: {mes_objetivo} ---")

# El bucle 'for' recorre los 3 SKUs de forma automática

for sku in df_agrupado['SKU'].unique():
    
    # 1. Aislamos los datos del SKU actual
    
    serie_sku = df_agrupado[df_agrupado['SKU'] == sku].copy()
    serie_sku['Mes_Anio'] = serie_sku['Mes_Anio'].dt.to_timestamp()
    serie_sku.set_index('Mes_Anio', inplace=True)
    serie_historica = serie_sku['Litros_Vendidos']
    
    # 2. Entrenamos el modelo matemático
    modelo = ExponentialSmoothing(serie_historica, trend='add', seasonal='add', seasonal_periods=12).fit()
    
    # 3. Proyectamos y extraemos específicamente el volumen de noviembre
    pronostico_futuro = modelo.forecast(6)
    volumen_proyectado = pronostico_futuro.loc[mes_objetivo]
    
    # 4. Guardamos el resultado en nuestro diccionario
    demanda_noviembre[sku] = round(volumen_proyectado)
    
    print(f"SKU: {sku} -> {demanda_noviembre[sku]:,.0f} Litros")

--- PRONÓSTICO DE DEMANDA: 2026-11-01 ---
SKU: LAV-BLA-03 -> 7,330 Litros
SKU: LAV-CLA-01 -> 28,113 Litros
SKU: LAV-FLO-02 -> 13,467 Litros


### Realizamos el MRP, conociendo las recetas y tiempos para la producción de cada articulo

In [104]:
# 1. Demanda dinámica (Viene de la Fase 2)
demanda_dinamica = demanda_noviembre

asignacion_tanques = {'LAV-CLA-01': 'TK-100', 'LAV-FLO-02': 'TK-200', 'LAV-BLA-03': 'TK-200'}
costo_hora_operario = 5000.0  
costo_kwh = 120.0             
horas_mezclado = 3.0          

# 2. Leemos AMBOS archivos maestros
df_equipos = pd.read_csv('maestro_equipos.csv')
df_bom = pd.read_csv('maestro_recetas.csv')  

fecha = pd.to_datetime(mes_objetivo)

print(f"--- ANÁLISIS FINANCIERO TOTAL (OPEX + MATERIA PRIMA) --- {fecha.strftime('%m/%Y')}")

costo_global_planta = 0

# ---> SE REALIZA UN LOOP PARA ARMAR UNA LISTA CON LOS LOTES A REALIZAR
lotes_a_programar = []

for sku, demanda in demanda_dinamica.items():
    # --- DATOS DEL EQUIPO ---
    tanque = asignacion_tanques[sku]
    info_tanque = df_equipos[df_equipos['ID_Tanque'] == tanque].iloc[0]
    capacidad = info_tanque['Capacidad_L']
    potencia_motor = info_tanque['Potencia_Agitador_kW']
    tiempo_lavado = info_tanque['Tiempo_CIP_hs']
    costo_quimicos_cip = info_tanque['Costo_CIP_ARS']
    
    lotes = math.ceil(demanda / capacidad)
    
    # --- 1. CÁLCULO DE OPEX (Fijo por Lote) ---
    horas_totales_batch = horas_mezclado + tiempo_lavado
    costo_labor_batch = horas_totales_batch * costo_hora_operario
    costo_energia_batch = horas_mezclado * potencia_motor * costo_kwh
    opex_batch = costo_labor_batch + costo_energia_batch + costo_quimicos_cip

    # ---> NUEVO: GUARDAMOS LOS LOTES FÍSICOS PARA LA FASE 4
    for i in range(lotes):
        lotes_a_programar.append({
            'Lote': f"{sku}_B{i+1}",
            'Tanque': tanque,
            'Duracion_hs': horas_totales_batch
        })
    
    # --- 2. CÁLCULO DE MATERIA PRIMA (Variable según receta) ---
    receta = df_bom[df_bom['SKU_Producto'] == sku]
    costo_mp_batch = 0
    
    # Iteramos sobre los ingredientes que lleva un tanque entero
    for idx, fila in receta.iterrows():
        litros_insumo = capacidad * fila['Fraccion_Receta']
        costo_insumo = litros_insumo * fila['Costo_Unitario_ARS']
        costo_mp_batch += costo_insumo

    # --- 3. COSTO TOTAL DEL BATCH ---
    costo_total_batch = opex_batch + costo_mp_batch
    inversion_total_sku = costo_total_batch * lotes
    costo_global_planta += inversion_total_sku
    
    print(f"\nSKU: {sku} | {lotes} Batch(es) en {tanque} ({capacidad} L c/u)")
    print(f"  * Costo Materia Prima (Reactivos): ${costo_mp_batch:,.2f} por lote")
    print(f"  * Costo OPEX (Energía, Labor, CIP): ${opex_batch:,.2f} por lote")
    print(f"  -> Costo Total de un Lote: ${costo_total_batch:,.2f}")
    print(f"  -> INVERSIÓN MENSUAL PARA CUBRIR DEMANDA: ${inversion_total_sku:,.2f}")

print("\n" + "="*60)
print(f"PRESUPUESTO TOTAL REQUERIDO PARA EL MES: ${costo_global_planta:,.2f}")
print(f"\n[Sistema] Se generaron {len(lotes_a_programar)} lotes en memoria listos para Pyomo.")
print(lotes_a_programar)

--- ANÁLISIS FINANCIERO TOTAL (OPEX + MATERIA PRIMA) --- 11/2026

SKU: LAV-BLA-03 | 2 Batch(es) en TK-200 (5000 L c/u)
  * Costo Materia Prima (Reactivos): $636,375.00 por lote
  * Costo OPEX (Energía, Labor, CIP): $37,380.00 por lote
  -> Costo Total de un Lote: $673,755.00
  -> INVERSIÓN MENSUAL PARA CUBRIR DEMANDA: $1,347,510.00

SKU: LAV-CLA-01 | 3 Batch(es) en TK-100 (10000 L c/u)
  * Costo Materia Prima (Reactivos): $717,500.00 por lote
  * Costo OPEX (Energía, Labor, CIP): $48,400.00 por lote
  -> Costo Total de un Lote: $765,900.00
  -> INVERSIÓN MENSUAL PARA CUBRIR DEMANDA: $2,297,700.00

SKU: LAV-FLO-02 | 3 Batch(es) en TK-200 (5000 L c/u)
  * Costo Materia Prima (Reactivos): $716,750.00 por lote
  * Costo OPEX (Energía, Labor, CIP): $37,380.00 por lote
  -> Costo Total de un Lote: $754,130.00
  -> INVERSIÓN MENSUAL PARA CUBRIR DEMANDA: $2,262,390.00

PRESUPUESTO TOTAL REQUERIDO PARA EL MES: $5,907,600.00

[Sistema] Se generaron 8 lotes en memoria listos para Pyomo.
[{'Lote':

### Optimización y Secuenciación de Planta con Pyomo

In [110]:
import pulp
import pandas as pd
import itertools

# =============================================================
# FASE 4: OPTIMIZACIÓN DE PLANTA CON PuLP (MILP)
# =============================================================

# 1. PREPARACIÓN DE DATOS (Lo que entra al algoritmo)
# Filtramos qué lotes van a qué tanque basándonos en la Fase 3.
lotes_tk100 = [item['Lote'] for item in lotes_a_programar if item['Tanque'] == 'TK-100']
lotes_tk200 = [item['Lote'] for item in lotes_a_programar if item['Tanque'] == 'TK-200']

# Armamos el diccionario de tiempos: {'LAV-CLA-01_B1': 5.0, ...}
duracion = {item['Lote']: item['Duracion_hs'] for item in lotes_a_programar}
todos_los_lotes = list(duracion.keys())

# 2. CREACIÓN DEL MODELO
# En PuLP, creamos el problema y directamente le decimos que queremos MINIMIZAR el tiempo.
modelo = pulp.LpProblem("Programacion_Lavandina", pulp.LpMinimize)

# 3. LAS VARIABLES DE DECISIÓN (Las "X" que el modelo debe averiguar)
# 'arranque': Un diccionario con el horario de inicio de cada lote. (lowBound=0 impide horas negativas)
arranque = pulp.LpVariable.dicts("Inicio", todos_los_lotes, lowBound=0, cat='Continuous')

# 'makespan': Una única variable para la hora en que se apaga la fábrica.
makespan = pulp.LpVariable("Makespan_Total", lowBound=0, cat='Continuous')

M = 100 # Constante para encender/apagar reglas lógicas

# 4. REGLAS FÍSICAS: EVITAR CHOQUES EN LOS TANQUES
# Lógica para el TK-100
if len(lotes_tk100) > 1:
    pares_100 = list(itertools.combinations(lotes_tk100, 2))
    # Creamos interruptores binarios (0 o 1) para decidir quién va primero en cada pareja
    y100 = pulp.LpVariable.dicts("Orden100", pares_100, cat='Binary')
    
    for i, j in pares_100:
        # En PuLP las reglas se agregan sumándolas al modelo con "+="
        # Esto dice: "El lote J arranca después de I" O BIEN "El lote I arranca después de J"
        modelo += arranque[j] >= arranque[i] + duracion[i] - M * (1 - y100[(i,j)])
        modelo += arranque[i] >= arranque[j] + duracion[j] - M * y100[(i,j)]

# Lógica para el TK-200
if len(lotes_tk200) > 1:
    pares_200 = list(itertools.combinations(lotes_tk200, 2))
    y200 = pulp.LpVariable.dicts("Orden200", pares_200, cat='Binary')
    
    for i, j in pares_200:
        modelo += arranque[j] >= arranque[i] + duracion[i] - M * (1 - y200[(i,j)])
        modelo += arranque[i] >= arranque[j] + duracion[j] - M * y200[(i,j)]

# 5. REGLA DEL MAKESPAN
# Obligamos a que el tiempo total de la fábrica sea mayor o igual al fin de cada lote procesado.
for lote in todos_los_lotes:
    modelo += makespan >= arranque[lote] + duracion[lote]

# 6. FUNCIÓN OBJETIVO
# Le indicamos al modelo que la meta final es hacer que esta variable sea lo más chica posible.
modelo += makespan

# 7. RESOLUCIÓN
print("Calculando el cronograma óptimo con PuLP...")
modelo.solve()

# =============================================================
# 8. EXTRACCIÓN DE DATOS Y GANTT
# =============================================================
cronograma = []

for lote in todos_los_lotes:
    tanque = 'TK-100' if lote in lotes_tk100 else 'TK-200'
    # Con pulp.value() extraemos el número calculado por el motor
    hora_inicio = pulp.value(arranque[lote])
    hora_fin = hora_inicio + duracion[lote]
    cronograma.append({'Tanque': tanque, 'Lote': lote, 'Inicio (hs)': hora_inicio, 'Fin (hs)': hora_fin})

df_gantt = pd.DataFrame(cronograma).sort_values(by=['Tanque', 'Inicio (hs)'])

print("\n--- CRONOGRAMA MAESTRO DE PRODUCCIÓN (MPS) ---")
print(df_gantt.to_string(index=False))
print("-" * 46)
print(f"TIEMPO TOTAL (Makespan): {pulp.value(makespan)} horas operativas.")

df_gantt.to_csv('cronograma_produccion_pulp.csv', index=False)

Calculando el cronograma óptimo con PuLP...
Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /home/eliseo/anaconda3/lib/python3.11/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/d5a2eb34eb134b31b5468e0062e970c5-pulp.mps -timeMode elapsed -solve -printingOptions all -solution /tmp/d5a2eb34eb134b31b5468e0062e970c5-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 39 COLUMNS
At line 161 RHS
At line 196 BOUNDS
At line 210 ENDATA
Problem MODEL has 34 rows, 22 columns and 94 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 5 - 0.00 seconds
Cgl0004I processed model has 34 rows, 22 columns (13 integer (13 of which binary)) and 94 elements
Cbc0038I Initial state - 13 integers unsatisfied sum - 0.6
Cbc0038I Pass   1: suminf.    0.00000 (0) obj. 22.5 iterations 15
Cbc0038I Solution found of 22.5
Cbc0038I Relaxing continuous gives 22.5
Cbc0